In [1]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

In [2]:
import matplotlib.pyplot as plt
import numpy as np
from monai.transforms import LoadImage, Resize, EnsureChannelFirst, Orientation, Compose, SaveImage, Transform
from monai.bundle import ConfigParser, download
from monai.data.meta_tensor import MetaTensor
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import SGD
import seaborn as sns
from ipywidgets import interact
import nibabel as nib
from nibabel import Nifti1Image
from totalsegmentator.python_api import totalsegmentator
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, random_split
import torch.optim as optim
import lightning as L
from lightning.pytorch import Trainer



In [3]:
%matplotlib inline

def show_scrollable_image(image, cmap="nipy_spectral"):
    """
    Displays a 3D array as a scrollable series of slices.
    
    Parameters:
    - image (numpy.ndarray): A 3D numpy array (e.g., (H, W, D)) representing the image.
    """
    
    # Define a function to display each slice
    def display_slice(slice_index):
        plt.figure(figsize=(8, 8))
        plt.imshow(image[:, :, slice_index], cmap=cmap)
        plt.colorbar(label='HU')
        plt.axis("off")
        plt.title(f"Slice {slice_index + 1} / {image.shape[2]}")
        plt.show()
    
    # Create an interactive slider to scroll through slices
    interact(display_slice, slice_index=(0, image.shape[2] - 1))

In [4]:
def nifti1image_to_metatensor(nifti_img: Nifti1Image) -> MetaTensor:
    """
    Converts a Nifti1Image object to a MetaTensor with metadata.
    Args:
        nifti_img (Nifti1Image): A loaded NIfTI image.
    Returns:
        MetaTensor: A MONAI MetaTensor containing the image data and metadata.
    """
    # Convert voxel data to PyTorch tensor
    image_tensor = torch.tensor(nifti_img.get_fdata(), dtype=torch.float32)

    # Extract metadata
    metadata = {
        "affine": torch.tensor(nifti_img.affine, dtype=torch.float32),  # Affine transformation matrix
        "spacing": nifti_img.header.get_zooms(),  # Voxel spacing (x, y, z)
        "original_shape": image_tensor.shape  # Store original shape for reference
    }

    # Create a MetaTensor with metadata
    meta_tensor = MetaTensor(image_tensor, meta=metadata)

    return meta_tensor

In [5]:
def resize_to_target_slices(image: MetaTensor, heart_mask: MetaTensor, target_width=512, target_height=512, target_slices=128):
    """
    Extracts the heart region from the input scan and resizes it to a target number of slices, 
    preserving metadata while correctly computing voxel spacing.

    Args:
        image (MetaTensor): The input 3D CT/MRI scan.
        heart_mask (MetaTensor): Binary mask indicating heart region.
        target_slices (int, optional): Number of slices to resize to. Default is 128.

    Returns:
        MetaTensor: Resized image with updated metadata.
    """
    if isinstance(heart_mask, MetaTensor):
        heart_mask = heart_mask.as_tensor().cpu().numpy()
    
    z_indices = np.any(heart_mask > 0, axis=(0, 1))
    heart_slices = np.where(z_indices)[0]
    # print(heart_slices)
    if len(heart_slices) == 0:
        raise ValueError("No heart region found in the mask.")
    
    start_slice = heart_slices[0]
    end_slice = heart_slices[-1] + 1
    
    sliced_image = image[:, :, start_slice:end_slice]
    if not isinstance(sliced_image, MetaTensor):
        sliced_image = MetaTensor(sliced_image, meta=image.meta)
    
    height, width, original_depth = image.shape
    # sliced_depth = sliced_image.shape[-1]

    sliced_image = sliced_image.unsqueeze(0)  # Shape becomes (1, H, W, Z)
    
    resizer = Resize(spatial_size=(target_width,target_height,target_slices), mode="trilinear", align_corners=True)
    resized_image = resizer(sliced_image)
    
    resized_image = resized_image.squeeze(0)

    # Update metadata
    new_meta = image.meta.copy()  
    
    original_spacing = image.meta.get("spacing", (1.0, 1.0, 1.0))
    new_spacing = (
        original_spacing[0],
        original_spacing[1],
        original_spacing[2] * (original_depth / target_slices)
    )
    new_meta["spacing"] = new_spacing

    # Keep affine transformation matrix but adjust slice resolution
    if "affine" in new_meta:
        new_meta["affine"][-1, -1] *= (original_depth / target_slices)  # Adjust Z scaling

    # Return resized MetaTensor with updated metadata
    return MetaTensor(resized_image, meta=new_meta)

In [6]:
def align_segmentation_to_image(segmentation):
    """
    Aligns the TotalSegmentator output (segmentation) with the original image
    by flipping the z-axis of the segmentation mask.
    
    Parameters:
        image (torch.Tensor or np.ndarray): Original image (H x W x D).
        segmentation (torch.Tensor or np.ndarray): TotalSegmentator output (H x W x D).
        
    Returns:
        torch.Tensor: Segmentation aligned with the image.
    """
   
    # Flip along the z-axis (last dimension)
    aligned_segmentation = torch.flip(segmentation, dims=[-1])
    
    return aligned_segmentation

# Example usage
# aligned_mask = align_segmentation_to_image(full_image)



In [7]:
def custom_name_formatter(metadict: dict, saver: Transform) -> dict:
    """Returns a kwargs dict for :py:meth:`FolderLayout.filename`,
    according to the input metadata and SaveImage transform."""
    subject = "heart_resized"
    patch_index = None
    return {"subject": f"{subject}", "idx": patch_index}

In [8]:
def preprocess_full_heart(input_file_path: str):
    # input_folder = f"../data/Inputs/{input_file_path}"
    dicom_file_path = os.path.join(input_file_path, "DICOM")
    subfolders = sorted([f for f in os.listdir(dicom_file_path) if os.path.isdir(os.path.join(dicom_file_path, f))])
    
    if not subfolders:
        raise FileNotFoundError(f"No subfolders found in {dicom_file_path}")
    
    first_folder = subfolders[0]
    dicom_file_path = os.path.join(dicom_file_path, first_folder)    

    output_folder = input_file_path.replace("Inputs", "Outputs")
    
    if os.path.exists(f"{output_folder}/heart_resized.nii.gz"):
        image_loader = LoadImage(image_only=True, ensure_channel_first=True)
        resized_image = image_loader(f"{output_folder}/heart_resized.nii.gz")
        resized_image = resized_image.squeeze(dim=-1)
        # resized_image = resize_to_target_slices(output_image, output_image)
        # resized_image = nib.load(f"{output_folder}/heart_resized.nii.gz")
    else:
        # output_image = totalsegmentator(dicom_file_path, output_folder, license_number="aca_BWYHC6UQQFDU8A", roi_subset=["heart"], device="mps")
        output_image = totalsegmentator(dicom_file_path, output_folder, roi_subset=["heart"], device="mps")
        output_image = nifti1image_to_metatensor(output_image)
        resized_image = resize_to_target_slices(output_image, output_image,target_height=256,target_width=256,target_slices=64)
        # nib.save(resized_image, f"{output_folder}/heart_resized.nii.gz")
        print("saving")
        image_saver = SaveImage(output_dir=f"{output_folder}", separate_folder=False, output_postfix="", output_name_formatter=custom_name_formatter)
        image_saver(resized_image)
        print("saved")
        print(resized_image.shape)
    
    return resized_image
    


# preprocess_full_heart("takotsubo_cases/testing_subset/LET 04456356")


In [9]:
x_data = []
y_labels = []

In [10]:
takotsubo_data_path = '../data/Inputs/takotsubo_cases/'

# takotsubo_preprocessed_data = []

for folder_name in sorted(os.listdir(takotsubo_data_path)):
    folder_path = os.path.join(takotsubo_data_path, folder_name)
    if os.path.isdir(folder_path):
        print(folder_path)
        preprocessed_heart = preprocess_full_heart(folder_path)
        x_data.append(preprocessed_heart)
        y_labels.append(1)


../data/Inputs/takotsubo_cases/AD 12191953
../data/Inputs/takotsubo_cases/AG 11370442
../data/Inputs/takotsubo_cases/AMH 36922300
../data/Inputs/takotsubo_cases/AMS 22722383
../data/Inputs/takotsubo_cases/APH 03516218
../data/Inputs/takotsubo_cases/BB 08903262
../data/Inputs/takotsubo_cases/BEH 04037842 with thrombus
../data/Inputs/takotsubo_cases/BFM 10624765
../data/Inputs/takotsubo_cases/BLT 11022191
../data/Inputs/takotsubo_cases/CAMcD 10747764 with thrombus
../data/Inputs/takotsubo_cases/CDA 38093134
../data/Inputs/takotsubo_cases/CLB 50002879
../data/Inputs/takotsubo_cases/CML 11113354
../data/Inputs/takotsubo_cases/CMS 04664231
../data/Inputs/takotsubo_cases/DCA 09939935
../data/Inputs/takotsubo_cases/DD 26362236
../data/Inputs/takotsubo_cases/DEM 09902248
../data/Inputs/takotsubo_cases/DIW 10053411
../data/Inputs/takotsubo_cases/DJD 06122410
../data/Inputs/takotsubo_cases/DJK 2493 RVH
../data/Inputs/takotsubo_cases/DJM 34687566
../data/Inputs/takotsubo_cases/ECW 33674557
../dat

In [11]:
normal_data_path = '../data/Inputs/normal_cases/'

# normal_preprocessed_data = []
for folder_name in sorted(os.listdir(normal_data_path)):
    folder_path = os.path.join(normal_data_path, folder_name)
    if os.path.isdir(folder_path):
        print(folder_path)
        preprocessed_heart = preprocess_full_heart(folder_path)
        x_data.append(preprocessed_heart)
        y_labels.append(0)

../data/Inputs/normal_cases/AAP 50415783_61F
../data/Inputs/normal_cases/AB 10470318_90F
../data/Inputs/normal_cases/AG 519880_37F
../data/Inputs/normal_cases/AIB 36444628_88F
../data/Inputs/normal_cases/AMB 08058471_74M
../data/Inputs/normal_cases/ASJ 05633540_68F
../data/Inputs/normal_cases/ASSA 39309711_56F
../data/Inputs/normal_cases/BA 37586690_62M
../data/Inputs/normal_cases/BJB 42898_78F
../data/Inputs/normal_cases/BMB 26209098_65M
../data/Inputs/normal_cases/BWB 08372559_71M
../data/Inputs/normal_cases/CAB 30473847_51F
../data/Inputs/normal_cases/CAD 35811017_49F
../data/Inputs/normal_cases/CAM 29734621_78F
../data/Inputs/normal_cases/CB 11607017_75F
../data/Inputs/normal_cases/CEEB 08099780_60F
../data/Inputs/normal_cases/CEK 02226835_80F
../data/Inputs/normal_cases/CMLB 50124639_67F
../data/Inputs/normal_cases/DAB 37356987_82F
../data/Inputs/normal_cases/DB 24229841_67F
../data/Inputs/normal_cases/DCE 22600738_50M
../data/Inputs/normal_cases/DEA 9230_52M
../data/Inputs/normal

In [ ]:
class CTScanDataset(Dataset):
    def __init__(self, images, labels):
        self.images = images.float().unsqueeze(1)  # (N, 1, H, W, D)
        self.labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)  # (N, 1)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

# Convert MetaTensors to regular tensors
x_data = torch.stack([x.as_tensor() for x in x_data])  # (N, 512, 512, 128)
y_labels = torch.tensor(y_labels, dtype=torch.float32)

# Create dataset
dataset = CTScanDataset(x_data, y_labels)

# Split dataset (80% train, 20% test)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

# train_dataset, test_dataset = random_split(dataset, [train_size, test_size])


# train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=0)
# test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)


/var/folders/jm/k37qm_7n7dj5fp2rnwhd32y80000gn/T/ipykernel_54383/1091880197.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)  # (N, 1)


In [ ]:
class CNN3D(L.LightningModule):
    def __init__(self, learning_rate=0.0001):
        super().__init__()
        self.learning_rate = learning_rate

        self.conv1 = nn.Conv3d(1, 16, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv3d(16, 32, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv3d(32, 64, kernel_size=3, stride=1, padding=1)

        self.global_pool = nn.AdaptiveAvgPool3d((4, 4, 4))

        self.fc1 = nn.Linear(64 * 4 * 4 * 4, 128)
        self.fc2 = nn.Linear(128, 1)

        self.criterion = nn.BCELoss()  # Binary classification loss

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.relu(self.conv2(x))
        x = self.pool(x)
        x = torch.relu(self.conv3(x))
        x = self.pool(x)

        x = self.global_pool(x)
        x = torch.flatten(x, start_dim=1)

        x = torch.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        return x

    def training_step(self, batch, batch_idx):
        images, labels = batch
        labels = labels.float()  # Ensure labels are float for BCELoss
        outputs = self(images)
        loss = self.criterion(outputs, labels)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        labels = labels.float()
        outputs = self(images)
        loss = self.criterion(outputs, labels)
        preds = (outputs > 0.5).float()
        acc = (preds == labels).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=self.learning_rate)

In [14]:
class CTScanDataModule(L.LightningDataModule):
    def __init__(self, dataset, batch_size=1, train_split=0.8):
        super().__init__()
        self.dataset = dataset
        self.batch_size = batch_size
        self.train_split = train_split

    def setup(self, stage=None):
        train_size = int(self.train_split * len(self.dataset))
        val_size = len(self.dataset) - train_size
        self.train_dataset, self.val_dataset = random_split(self.dataset, [train_size, val_size])

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=0)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=0)


In [ ]:

# Initialize Model & DataModule
model = CNN3D()
data_module = CTScanDataModule(dataset, batch_size=1)

# Set up Trainer
trainer = Trainer(max_epochs=10, accelerator="auto", log_every_n_steps=1)

# Train Model
trainer.fit(model, datamodule=data_module)


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/awabakram/git/Takotsubo-Syndrome/myenv3/lib/python3.9/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default

  | Name        | Type              | Params | Mode 
----------------------------------------------------------
0 | conv1       | Conv3d            | 448    | train
1 | pool        | MaxPool3d         | 0      | train
2 | conv2       | Conv3d            | 13.9 K | train
3 | conv3       | Conv3d            | 55.4 K | train
4

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/Users/awabakram/git/Takotsubo-Syndrome/myenv3/lib/python3.9/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/Users/awabakram/git/Takotsubo-Syndrome/myenv3/lib/python3.9/site-packages/torch/nn/functional.py:882: UserWarning: The operator 'aten::max_pool3d_with_indices' is not currently supported on the MPS backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at /private/var/folders/jm/k37qm_7n7dj5fp2rnwhd32y80000gn/T/pip-req-build-n_0s720w/aten/src/ATen/mps/MPSFallback.mm:13.)
  return torch.max_pool3d(input, kernel_size, stride, padding, dilation, ceil_mode)
/Users/awabakram/git/Takotsubo-Syndrome/myenv3/lib/python3.9/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataload

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=20` reached.
